<a href="https://colab.research.google.com/github/prameesha04/Agentic_AI_Internship/blob/main/BERT_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torchvision
! pip uninstall datasets
! pip install transformers datasets torch scikit-learn

Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Would remove:
    /usr/local/bin/datasets-cli
    /usr/local/lib/python3.12/dist-packages/datasets-4.8.5.dist-info/*
    /usr/local/lib/python3.12/dist-packages/datasets/*
Proceed (Y/n)? y
  Successfully uninstalled datasets-4.8.5
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification,Trainer, TrainingArguments
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
dataset=load_dataset('stanfordnlp/imdb')
dataset=dataset.shuffle(seed=42)
small_train=dataset['train'].select(range(500))
small_test=dataset['test'].select(range(200))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
small_train

Dataset({
    features: ['text', 'label'],
    num_rows: 500
})

In [ ]:
small_test

Dataset({
    features: ['text', 'label'],
    num_rows: 200
})

In [ ]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
def tokenizer_function(examples):
  return tokenizer(examples['text'],padding='max_length',truncation=True)
train_enc=small_train.map(tokenizer_function,batched=True)
test_enc=small_test.map(tokenizer_function,batched=True)


In [ ]:
train_enc

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [ ]:
train_enc.set_format('torch',columns=['input_ids','attention_mask','label'])
test_enc.set_format('torch',columns=['input_ids','attention_mask','label'])


In [ ]:
model=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Define the evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    labels = labels.astype('int')
    preds = preds.astype('int')
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': float(acc),
        'f1': float(f1),
        'precision': float(precision),
        'recall': float(recall)
    }

In [ ]:
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive'

In [ ]:
training_args=TrainingArguments(
    output_dir='.results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    num_train_epochs=2,
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

In [ ]:

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.669811,0.660000,0.514286,0.818182,0.375000
2,No log,0.506270,0.840000,0.822222,0.880952,0.770833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=126, training_loss=0.640761239188058, metrics={'train_runtime': 125.2628, 'train_samples_per_second': 7.983, 'train_steps_per_second': 1.006, 'total_flos': 263111055360000.0, 'train_loss': 0.640761239188058, 'epoch': 2.0})

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs=tokenizer("Some input",return_tensors='pt').to(device)
outputs=model(**inputs)

In [ ]:
text='This is  good movie'
prediction=torch.argmax(outputs.logits).item()
print('Predicted Sentiments','Positive' if prediction==1 else 'Negative')

Predicted Sentiments Positive
